# 🤖 Step 3: Model Training (Random Forest)

Train a Random Forest classifier using GPS-labeled ground truth points.

## What This Notebook Does:
- ✅ Auto-generate Class 5 (Landmass) training points from high-NDVI areas
- ✅ Merge with existing ground truth CSV
- ✅ Extract 7-band spectral features at GPS locations
- ✅ Train Random Forest Classifier (100 trees)
- ✅ Evaluate performance (accuracy, confusion matrix, feature importance)
- ✅ Save trained model and scaler

## Training Data:
- **Source**: GPS-labeled points from field surveys (`Ground_TruthFinal.csv`)
- **Auto-augmentation**: ~100 Landmass points generated automatically
- **Classes**: 5 (Seagrass=1, Sand=2, Seaweed=3, Water=4, Landmass=5)
- **Features per point**: 7 (B02, B03, B04, B08, NDVI, NDWI, Texture)

## Model Details:
- **Algorithm**: Random Forest (ensemble of decision trees)
- **n_estimators**: 100 trees
- **max_depth**: 15 levels
- **Training time**: ~5-10 seconds on CPU

## Output:
- `../models/coastal_classifier_model.pkl` - Trained model
- `../models/feature_scaler.pkl` - Standardization scaler
- `../models/model_metadata.json` - Accuracy, classes, feature names

---

**Previous:** [02_preprocessing.ipynb](02_preprocessing.ipynb)  
**Next:** [04_prediction_analysis.ipynb](04_prediction_analysis.ipynb)

In [1]:
# ============================================================
# AUTO-GENERATE CLASS 5 (Landmass) TRAINING POINTS
# Add this cell to 02_preprocessing.ipynb or before training
# ============================================================

import rasterio
import numpy as np
import pandas as pd
from rasterio.warp import transform as rio_transform

print("🌿 Generating Class 5 (Landmass/Terrestrial Vegetation) training points...")

# Load the processed image
with rasterio.open("../data/processed/processed_image_with_indices.tif") as src:
    image_data = src.read()
    profile = src.profile
    transform = src.transform
    crs = src.crs

# Extract the index bands
# Band layout: [B02, B03, B04, B08, NDVI, NDWI, Texture] → indices 0-6
ndvi = image_data[4]
ndwi = image_data[5]

# ---------------------------------------------------------------
# LANDMASS CRITERIA:
#   - High NDVI  (> 0.35) → Dense vegetation, definitely not ocean
#   - Low  NDWI  (< -0.1) → Not water, not coastal wetland
#   - Both conditions together = terrestrial landmass
# ---------------------------------------------------------------
landmass_mask = (ndvi > 0.35) & (ndwi < -0.1)

print(f"   Candidate landmass pixels found: {np.sum(landmass_mask):,}")

# Get pixel coordinates of all candidates
rows, cols = np.where(landmass_mask)

# ---------------------------------------------------------------
# SMART SAMPLING: Don't just take random pixels.
# Sample from a spatial grid to ensure geographic spread.
# ---------------------------------------------------------------
N_POINTS = 100  # Adjust: 50 minimum, 150 maximum recommended

if len(rows) < N_POINTS:
    print(f"⚠️  Only {len(rows)} candidate pixels found. Using all of them.")
    sampled_rows, sampled_cols = rows, cols
else:
    # Spatially-distributed sampling using a grid approach
    # Divide the image into a grid and sample uniformly from each cell
    n_grid = int(np.ceil(np.sqrt(N_POINTS)))
    h, w = ndvi.shape
    grid_h = h // n_grid
    grid_w = w // n_grid
    
    sampled_rows, sampled_cols = [], []
    
    for gi in range(n_grid):
        for gj in range(n_grid):
            # Find landmass pixels within this grid cell
            r_min, r_max = gi * grid_h, (gi + 1) * grid_h
            c_min, c_max = gj * grid_w, (gj + 1) * grid_w
            
            in_cell = (rows >= r_min) & (rows < r_max) & \
                      (cols >= c_min) & (cols < c_max)
            
            cell_rows = rows[in_cell]
            cell_cols = cols[in_cell]
            
            if len(cell_rows) > 0:
                # Pick 1 random pixel from this grid cell
                idx = np.random.randint(len(cell_rows))
                sampled_rows.append(cell_rows[idx])
                sampled_cols.append(cell_cols[idx])
    
    sampled_rows = np.array(sampled_rows)
    sampled_cols = np.array(sampled_cols)
    
    print(f"   Spatially sampled: {len(sampled_rows)} points across {n_grid}×{n_grid} grid")

# ---------------------------------------------------------------
# CONVERT pixel (row, col) → geographic (Lon, Lat)
# ---------------------------------------------------------------
# rasterio xy() returns coordinates in the image's CRS
xs, ys = rasterio.transform.xy(transform, sampled_rows, sampled_cols)

# Convert from image CRS → WGS84 (EPSG:4326) to match your CSV format
lons, lats = rio_transform(crs, 'EPSG:4326', xs, ys)

# ---------------------------------------------------------------
# BUILD THE NEW TRAINING ROWS
# ---------------------------------------------------------------
new_points_df = pd.DataFrame({
    'Longitude': lons,
    'Latitude':  lats,
    'Value':     5  # Class 5 = Landmass/Terrestrial Vegetation
})

print(f"\n✅ Generated {len(new_points_df)} Class 5 training points")
print(f"   Lon range: {min(lons):.4f} → {max(lons):.4f}")
print(f"   Lat range: {min(lats):.4f} → {max(lats):.4f}")

# ---------------------------------------------------------------
# MERGE WITH YOUR EXISTING GROUND TRUTH CSV
# ---------------------------------------------------------------
existing_csv_path = "../data/training/trainingdata/Ground_TruthFinal.csv"
existing_df = pd.read_csv(existing_csv_path)

print(f"\n📋 Existing training data: {len(existing_df)} points")
print(f"   Classes before: {sorted(existing_df['Value'].unique())}")

# Append the new class 5 points
combined_df = pd.concat([existing_df, new_points_df], ignore_index=True)

print(f"\n📋 Combined training data: {len(combined_df)} points")
print(f"   Classes after:  {sorted(combined_df['Value'].unique())}")
print(f"   Class distribution:")
for cls in sorted(combined_df['Value'].unique()):
    count = (combined_df['Value'] == cls).sum()
    names = {1:"Seagrass", 2:"Sand", 3:"Seaweed", 4:"Water", 5:"Landmass"}
    print(f"      Class {cls} ({names.get(cls,'Unknown')}): {count} samples")

# Save augmented CSV (keeps original safe)
augmented_path = "../data/training/trainingdata/Ground_TruthFinal_with_landmass.csv"
combined_df.to_csv(augmented_path, index=False)
print(f"\n💾 Saved to: {augmented_path}")
print("   ℹ️  Original CSV untouched. Use augmented file in notebook 03.")

🌿 Generating Class 5 (Landmass/Terrestrial Vegetation) training points...
   Candidate landmass pixels found: 38,570
   Spatially sampled: 33 points across 10×10 grid

✅ Generated 33 Class 5 training points
   Lon range: 120.6030 → 120.6276
   Lat range: 13.9515 → 13.9772

📋 Existing training data: 1056 points
   Classes before: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

📋 Combined training data: 1089 points
   Classes after:  [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
   Class distribution:
      Class 1 (Seagrass): 383 samples
      Class 2 (Sand): 246 samples
      Class 3 (Seaweed): 357 samples
      Class 4 (Water): 70 samples
      Class 5 (Landmass): 33 samples

💾 Saved to: ../data/training/trainingdata/Ground_TruthFinal_with_landmass.csv
   ℹ️  Original CSV untouched. Use augmented file in notebook 03.


## Load Libraries and Data

In [2]:
import pandas as pd
import numpy as np
import rasterio
from rasterio.warp import transform
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import joblib

# 1. Paths
image_path = "../data/processed/processed_image_with_indices.tif"
# Change 1: Point to the new CSV
csv_path = "../data/training/trainingdata/Ground_TruthFinal_with_landmass.csv"

# Change 2: Include class 5
INCLUDED_CLASSES = [1, 2, 3, 4, 5]

# 2. Load CSV
df = pd.read_csv(csv_path)

# Remove rows with NaN values in the 'Value' column
df = df.dropna(subset=['Value'])

print(f"📍 Original CSV classes: {sorted(df['Value'].unique())}")

# ⭐ STRATEGY: Keep coastal + add vegetation/landmass class for training
# Include vegetation classes if they exist (common codes: 5, 6, 7, etc.)
# Adjust INCLUDED_CLASSES based on your actual ground truth labels
INCLUDED_CLASSES = [1, 2, 3, 4, 5]  # Add landmass/veg class codes here
df = df[df['Value'].isin(INCLUDED_CLASSES)]

print(f"📍 Loaded CSV with {len(df)} valid labels")
print(f"   Classes included: {sorted(df['Value'].unique())}")
print(f"   Class distribution:")
for cls in sorted(df['Value'].unique()):
    count = (df['Value'] == cls).sum()
    class_name = {
        1: "Seagrass",
        2: "Sand",
        3: "Seaweed",
        4: "Water",
        5: "Landmass", 
    }.get(cls, f"Class {cls}")
    print(f"      Class {cls} ({class_name}): {count} samples")

# 3. Extract training data from image at ground truth locations
print("\n📍 Extracting training data from satellite image...")

X = []
y = []

with rasterio.open(image_path) as src:
    img_data = src.read()
    img_crs = src.crs
    
    for i, row in df.iterrows():
        # Get raw coords from CSV
        lon, lat = row['Longitude'], row['Latitude']
        
        # Convert coordinates to the image's coordinate system
        xs, ys = transform('EPSG:4326', img_crs, [lon], [lat])
        x_img, y_img = xs[0], ys[0]
        
        # Get pixel index
        r, c = src.index(x_img, y_img)
        
        # Check if pixel is valid and inside the image
        if 0 <= r < src.height and 0 <= c < src.width:
            val = img_data[:, r, c]
            # Only use pixels without NaN values
            if not np.isnan(val).any():
                X.append(val)
                y.append(int(row['Value']))

X, y = np.array(X), np.array(y)
print(f"\n✅ Extraction Complete! Found {len(X)} valid training points.")
print(f"   Features per point: {X.shape[1]}")
print(f"   Final class distribution:")
for cls in np.unique(y):
    count = np.sum(y == cls)
    percentage = (count / len(y)) * 100
    print(f"      Class {cls}: {count} samples ({percentage:.1f}%)")

📍 Original CSV classes: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
📍 Loaded CSV with 1089 valid labels
   Classes included: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
   Class distribution:
      Class 1 (Seagrass): 383 samples
      Class 2 (Sand): 246 samples
      Class 3 (Seaweed): 357 samples
      Class 4 (Water): 70 samples
      Class 5 (Landmass): 33 samples

📍 Extracting training data from satellite image...

✅ Extraction Complete! Found 1089 valid training points.
   Features per point: 7
   Final class distribution:
      Class 1: 383 samples (35.2%)
      Class 2: 246 samples (22.6%)
      Class 3: 357 samples (32.8%)
      Class 4: 70 samples (6.4%)
      Class 5: 33 samples (3.0%)


## Preparing Data for CNN

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split data into training and testing sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"📊 Data Split:")
print(f"  Training samples: {len(X_train)}")
print(f"  Testing samples: {len(X_test)}")
print(f"  Feature dimensions: {X_train.shape[1]}")

# Standardize features (important for many ML algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data prepared and scaled!")

📊 Data Split:
  Training samples: 871
  Testing samples: 218
  Feature dimensions: 7
✅ Data prepared and scaled!


## Train the Model

In [4]:
# Build and train Random Forest Classifier
# (More efficient than CNN for small datasets like yours)
print("🤖 Training Random Forest Classifier...")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Train the model
model.fit(X_train_scaled, y_train)

print("✅ Model training complete!")
print(f"   Trained with {len(X_train)} samples")
print(f"   Classes: {np.unique(y_train)}")

🤖 Training Random Forest Classifier...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:    0.2s


✅ Model training complete!
   Trained with 871 samples
   Classes: [1 2 3 4 5]


[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.8s finished


## Evaluate the Performance


In [5]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

# Make predictions on train and test sets
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Calculate metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, average='weighted', zero_division=0)
recall = recall_score(y_test, y_test_pred, average='weighted', zero_division=0)
f1 = f1_score(y_test, y_test_pred, average='weighted', zero_division=0)

print("=" * 60)
print("📊 MODEL PERFORMANCE EVALUATION")
print("=" * 60)
print(f"\n🎯 Accuracy:")
print(f"   Training: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"   Testing:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

print(f"\n📈 Testing Metrics:")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")

print(f"\n📋 Confusion Matrix (Test Set):")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

print(f"\n📝 Classification Report:")
print(classification_report(y_test, y_test_pred, zero_division=0))

# Feature importance
feature_importance = model.feature_importances_
print(f"\n🔍 Top 3 Most Important Features:")
top_features = np.argsort(feature_importance)[-3:][::-1]
features = ['B02 (Blue)', 'B03 (Green)', 'B04 (Red)', 'B08 (NIR)', 'NDVI', 'NDWI', 'Texture']
for idx, feat_idx in enumerate(top_features, 1):
    print(f"   {idx}. {features[feat_idx]}: {feature_importance[feat_idx]:.4f}")

print("\n✅ Evaluation Complete!")

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.0s finished


📊 MODEL PERFORMANCE EVALUATION

🎯 Accuracy:
   Training: 0.9793 (97.93%)
   Testing:  0.8716 (87.16%)

📈 Testing Metrics:
   Precision: 0.8739
   Recall:    0.8716
   F1-Score:  0.8706

📋 Confusion Matrix (Test Set):
[[68  1  8  0  0]
 [ 2 45  2  0  0]
 [ 9  2 60  0  0]
 [ 0  0  1 13  0]
 [ 0  3  0  0  4]]

📝 Classification Report:
              precision    recall  f1-score   support

           1       0.86      0.88      0.87        77
           2       0.88      0.92      0.90        49
           3       0.85      0.85      0.85        71
           4       1.00      0.93      0.96        14
           5       1.00      0.57      0.73         7

    accuracy                           0.87       218
   macro avg       0.92      0.83      0.86       218
weighted avg       0.87      0.87      0.87       218


🔍 Top 3 Most Important Features:
   1. B03 (Green): 0.2236
   2. B04 (Red): 0.1875
   3. Texture: 0.1684

✅ Evaluation Complete!


## Save the Model

In [6]:
import os
import json

# Create directories if needed
os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

# Save the trained model
model_path = '../models/coastal_classifier_model.pkl'
joblib.dump(model, model_path)
print(f"✅ Model saved to: {model_path}")

# Save the scaler (important for preprocessing future data)
scaler_path = '../models/feature_scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"✅ Scaler saved to: {scaler_path}")

# Save model metadata
metadata = {
    'model_type': 'RandomForestClassifier',
    'n_features': int(X_train.shape[1]),
    'feature_names': ['B02 (Blue)', 'B03 (Green)', 'B04 (Red)', 'B08 (NIR)', 'NDVI', 'NDWI', 'Texture'],
    'classes': [int(c) for c in np.unique(y)],
    'test_accuracy': float(test_accuracy),
    'training_samples': int(len(X_train)),
    'testing_samples': int(len(X_test))
}
metadata_path = '../models/model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Metadata saved to: {metadata_path}")

print("\n📦 Model Package Complete!")
print("   Ready for predictions in the next notebook.")

✅ Model saved to: ../models/coastal_classifier_model.pkl
✅ Scaler saved to: ../models/feature_scaler.pkl
✅ Metadata saved to: ../models/model_metadata.json

📦 Model Package Complete!
   Ready for predictions in the next notebook.
